In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector

In [2]:
def build_feature_map(n_qubits=8):
    x = Parameter("x")
    qc = QuantumCircuit(n_qubits, name="FeatureMap")
    for q in range(n_qubits):
        qc.rx((q * x) / 2, q)
    return qc

def build_HEA(n_qubits=8, seed=1):
    depth = 5
    rng = np.random.default_rng(seed=seed)
    theta = rng.uniform(0, 2 * np.pi, size=(depth, n_qubits, 2))
    
    qc = QuantumCircuit(n_qubits, name="HEA")
    for i in range(depth):
        for q in range(n_qubits):
            qc.rx(theta[i, q, 0], q)
            qc.rz(theta[i, q, 1], q)
        for q in range(n_qubits - 1):
            qc.cx(q, q+1)
    return qc

def build_kernel_feature_map(n_qubits=8):
    """Combines your teammate's HEA and Feature Map layers into the full map."""
    qc = QuantumCircuit(n_qubits, name="KernelMap")
    qc.compose(build_HEA(n_qubits), inplace=True)
    qc.compose(build_feature_map(n_qubits), inplace=True)
    return qc

In [3]:
def teammate_quantum_kernel(x, y, n_qubits=8):
    """
    Evaluates your teammate's exact circuit architecture using fast 
    Statevector overlap calculations for SVR differential elements.
    """
    qc = build_kernel_feature_map(n_qubits)
    # Extract the symbolic parameter 'x' embedded by the feature map
    param_x = [p for p in qc.parameters if p.name == "x"][0]
    
    # Bind individual scalar coordinates
    qc_x = qc.assign_parameters({param_x: x})
    qc_y = qc.assign_parameters({param_x: y})
    
    psi_x = Statevector.from_instruction(qc_x)
    psi_y = Statevector.from_instruction(qc_y)
    
    # Compute fidelity overlap: |<psi(y)|psi(x)>|^2
    return np.abs(np.vdot(psi_y.data, psi_x.data))**2

In [4]:
def dkernel_dx(kernel_fn, x, y, eps=1e-5):
    return (kernel_fn(x + eps, y) - kernel_fn(x - eps, y)) / (2 * eps)

def dkernel_dy(kernel_fn, x, y, eps=1e-5):
    return (kernel_fn(x, y + eps) - kernel_fn(x, y - eps)) / (2 * eps)

def dkernel_dxdy(kernel_fn, x, y, eps=1e-5):
    f_p_p = kernel_fn(x + eps, y + eps)
    f_p_m = kernel_fn(x + eps, y - eps)
    f_m_p = kernel_fn(x - eps, y + eps)
    f_m_m = kernel_fn(x - eps, y - eps)
    return (f_p_p - f_p_m - f_m_p + f_m_m) / (4 * eps**2)

class QKSVRSolverODE:
    def __init__(self, x_train, kernel_fn, gamma=1e5):
        """
        Support Vector Regression Solver driven exclusively by your teammate's kernel.
        """
        self.x_train = np.array(x_train)
        self.kernel_fn = kernel_fn
        self.gamma = gamma  
        self.N = len(self.x_train)
        
        # SVR Weight Matrix variables
        self.alpha = None
        self.beta = None
        self.eta = None
        self.b = None

    def fit(self, x0=0.0, f0=0.0):
        N = self.N
        X = self.x_train
        
        Omega_11 = np.zeros((N, N))
        Omega_10 = np.zeros((N, N))
        Omega_01 = np.zeros((N, N))
        Omega_00 = np.zeros((N, N))
        
        for i in range(N):
            for j in range(N):
                Omega_11[i, j] = dkernel_dxdy(self.kernel_fn, X[j], X[i])
                Omega_10[i, j] = dkernel_dx(self.kernel_fn, X[j], X[i])
                Omega_01[i, j] = dkernel_dy(self.kernel_fn, X[j], X[i])
                Omega_00[i, j] = self.kernel_fn(X[j], X[i])
                
        h_1 = np.array([dkernel_dx(self.kernel_fn, X[i], x0) for i in range(N)])
        h_0 = np.array([self.kernel_fn(X[i], x0) for i in range(N)])
        h_00 = self.kernel_fn(x0, x0)
        
        I_reg = np.eye(N) / self.gamma
        
        row1_left = Omega_11 + I_reg + Omega_10 + Omega_01 + Omega_00
        row1_mid = (h_1 + h_0).reshape(-1, 1)
        row1_right = np.ones((N, 1))
        
        row2_left = (h_1 + h_0).reshape(1, -1)
        row2_mid = np.array([[h_00 + (1.0 / self.gamma)]])
        row2_right = np.array([[1.0]])
        
        row3_left = np.ones((1, N))
        row3_mid = np.array([[1.0]])
        row3_right = np.array([[0.0]])
        
        A_top = np.hstack((row1_left, row1_mid, row1_right))
        A_mid = np.hstack((row2_left, row2_mid, row2_right))
        A_bot = np.hstack((row3_left, row3_mid, row3_right))
        A = np.vstack((A_top, A_mid, A_bot))
        
        B = np.zeros(N + 2)
        B[:N] = np.sin(X)
        B[N] = f0
        
        weights = np.linalg.solve(A, B)
        
        self.alpha = weights[:N]
        self.beta = weights[N]
        self.b = weights[N + 1]
        self.eta = self.alpha.copy()

    def predict(self, x_test, x0=0.0):
        predictions = []
        for x in x_test:
            term_alpha = sum(self.alpha[i] * dkernel_dx(self.kernel_fn, self.x_train[i], x) for i in range(self.N))
            term_eta = sum(self.eta[i] * self.kernel_fn(self.x_train[i], x) for i in range(self.N))
            term_beta = self.beta * self.kernel_fn(x0, x)
            
            f_val = term_alpha + term_eta + term_beta + self.b
            predictions.append(f_val)
        return np.array(predictions)

In [ ]:
# Training setup
x_train = np.linspace(0, 5, 25)
x_test = np.linspace(0, 5, 300)

# Target analytical solution
exact = 0.5 * (np.sin(x_test) - np.cos(x_test) + np.exp(-x_test))

# Instantiating and executing the SVR solver using your teammate's kernel
qk_svr = QKSVRSolverODE(x_train, teammate_quantum_kernel, gamma=1e6)
qk_svr.fit(x0=0.0, f0=0.0)
qk_pred = qk_svr.predict(x_test)

# Plotting the results
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.plot(x_test, exact, 'k-', label='Exact Analytical Solution', linewidth=2.5)
ax.plot(x_test, qk_pred, 'r--', label="SVR with Sacchyam's HEA Quantum Kernel (for 8 Qubits)", linewidth=2)


ax.set_title("ODE Integration Framework via Support Vector Regression", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Domain Grid x", fontsize=12)
ax.set_ylabel("f(x)", fontsize=12)
ax.legend(loc="best", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()